In [ ]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import pathlib
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

In [ ]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [ ]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
dev = tf.config.list_physical_devices()
print('Physical Devices : ', dev)

tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
dev = tf.config.list_logical_devices()
print('Available Devices : ', dev)

# Chapter 11: Deep Learning Text

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

In this section we look at the particular types of deep network architectures that work well when processing textual time series, as
well as other aspects specific to preparing and processing textual input.

## 11.3 Two Approaches for Representing Groups of Words: Sets and sequences

- Simplest approach: discard order and treat as unordered set: **bag-of-words models**
- Process strictly in order they appear, like steps in a timeseries **sequence models**
- Hybrid approach: **Transformer architecture** technically order-agnostic, yet injects word-postiion info into representations.

In this section we'll return to the IMDB movie reviews dataset.  We'll demonstrate each approach (bag-of-words and sequence models) on
this dataset and see how they do.

### 11.3.1 Preparing the IMDB movie reviews data

Though we are using the same dataset, for practice on the datapreprocessing and doing vectorization, we will get
the raw dataset and approach it as a new text-classification problem.

You can download the dataset from the following url.  If using our class DevContainers, the following expects the dataset to be
untared into the `../data/aclImdb/` directory, using the following commands. The `.tar.gz` file containing the data here is 81M in size,
and it extracts to a size of 433M total: 

```bash
vscode ➜ /workspaces/nndl/data (main) $ curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  7852k      0  0:00:10  0:00:10 --:--:-- 11.1M

vscode ➜ /workspaces/nndl/data (main) $ tar -xf aclImdb_v1.tar.gz 

vscode ➜ /workspaces/nndl/data (main) $ tar -xf aclImdb_v1.tar.gz 
433M    aclImdb

# we don't need the unsup data in train, get rid of it for space
vscode ➜ /workspaces/nndl/data (main) $ rm -rf aclImdb/train/unsup
```

This directory has a subdirectory structure you should be familair with:

```bash
vscode ➜ /workspaces/nndl/data (main) $ tree -d aclImdb/
aclImdb/
├── test
│   ├── neg
│   └── pos
└── train
    ├── neg
    └── pos
```


That is to say there is a corpus of textual reviews split into training and testing data.  This is a binary classification task, as you may recall, with an
even split of positive and negative reviews.  For instance the `train/pos/` subdirectory contains a set of 12,500 (plain ascii) text files, each of
which contains the text body of a positive-sentiment movie review.

It would be informative to look at a few reviews:

```bash
vscode ➜ /workspaces/nndl/data (main) $ cat aclImdb/train/pos/4077_10.txt 
I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drink and food in reach so you don't have to stop the film.<br /><br />
```

Let's prepare a validation set by setting apart 20% of the training text files in a new directory, aclImdb/val:

In [ ]:
base_dir = pathlib.Path("../data/aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"

# if the val_dir exists, assume that this code has already run and don't select validation set again
# WARNING: the files are actually moved from train to validation.  So running this multiple times
# is not something you want to do.
if not os.path.exists(val_dir):
    for category in ("neg", "pos"):
        print(f"selecting {category} validation reviews: ", end='')
        # make sure the destination directory is there for the copying
        os.makedirs(val_dir / category)
        # get list of all current training files
        files = os.listdir(train_dir / category)
    
        # shuffle the list of training files using a seed, to ensure we get the same validation
        # set every time we run the code.
        random.Random(1337).shuffle(files)
        # selection 20% to copy over for validation purposes
        num_val_samples = int(0.2 * len(files))
        val_files = files[-num_val_samples:]
        
        for fname in val_files:
            shutil.move(train_dir / category / fname, val_dir / category / fname)
            print('.', end='')
        print('')

As hinted at, we have a similar subdirectory structure, so we can use a similar utility method from keras for streaming text
datasets called `text_dataset_from_directory`.  Let's create three `Dataset` objects for training, validation and testing:

**Note**: good idea to verify you get 20,000 files from train, 5,000 from validation after split and full 25,000 still in test datasets here.

In [ ]:
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    train_dir, batch_size=batch_size
)

val_ds = keras.utils.text_dataset_from_directory(
    val_dir, batch_size=batch_size
)

test_ds = keras.utils.text_dataset_from_directory(
    test_dir, batch_size=batch_size
)

These datasets yield inputs that are TensorFlow `tf.string` tensors, and targets are `int32` tensors encoding values of
"0" or "1".

In [ ]:
for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

The inputs have not been tokenized or vectorized yet, they are just long strings here.

Notice that the 0'th input is labeled 0.  I think that because `neg` subdirectory name comes pefore `pos` subdirectory name, that we get the usual
encoding of negative reviews to 0 and positivie to 1.  I suspect that the `keras.utils` to stream from directories have an option to map subdirectory
names to desired label, or at a minimum you could always rename your directorys like `0-neg`, `1-pos` so they end up in order you want the
target labels to be assigned.

### 11.3.2 Processing words as a set: The bag-of-words approach

The simplest way to encode a piece of text for processing by a machine learning
model is to discard order and treat it as a set (a "bag") of tokens.  

You could either look at individual words (unigrams) or try to recover some local order information
by looking at groups of consecutive tokens (N-gram).

**Note**: You may recall that in our first example using this IMDB pos/neg binary classification that we multi-hot encoded the input texts into a vector of 10,000
features.  This was basically a unigram encoding into a set (the word was present or not present somewhere in the review).

#### Single words (unigrams) with binary encoding

If we use a bag of single words, the sentence "the cat sat on the mat" becomes

```
{"cat", "mat", "on", "sat", "the"}
```

The main advantage of this encoding is that you can represent an entire text as a single
vector, where each entry is a presence indicator for a given word. For instance,
using binary encoding (multi-hot), you’d encode a text as a vector with as many
dimensions as there are words in your vocabulary—with 0s almost everywhere and
some 1s for dimensions that encode words present in the text. This is what we did
when we worked with text data in chapters 4 and 5.

Let's process our raw text datasets with a `TextVectorization` layer so that they
yield multi-hot encoded binary word vectors.  Our layer will only look at 
single words (that is to say, **unigrams**).

In [ ]:
# limit vocabulary to 20,000 most frequent words, twice more than last time
# In general, 20,000 is about the right vocabulary size for text classification on
# a real world corpus
#
# also notice the parameter to specify encoding as multi-hot binary vectors
text_vectorization = layers.TextVectorization(
    max_tokens=20000,
    output_mode="multi_hot",
)

# prepare a dataset that only yields raw text inputs (no labels)
text_only_train_ds = train_ds.map(lambda x, y: x)
# use that dataset to index the dataset vocabulary via the adapt() method,
# so like before this initializes our text_vectorization instance with the training 
# corpus vocabulary
text_vectorization.adapt(text_only_train_ds)

# prepare processed versions of training, validation and test dataset
# e.g. these datasets produce binary encoded vectors of 1-gram input
# make sure to specify num_parallel_calls to leverage multi CPU cores
binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

Let's inspect the resulting TensorFlow `Dataset` to again make sure we understood the preprocessing that has just taken place.

If you recall, the `Dataset` class can be treated as an iterator, so we can ask it to give us the first batch of inputs
and labels like this.  Do you know what we should be expecting from iterating over this dataset?:

In [ ]:
for inputs, targets in binary_1gram_train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

We set the batch size to 32.  And the mapping we did in the previous cell causes the vectorization to happen, so that the
variably sized text reviews are mapped as 1-grams into a 20000 shaped vector with a 1 at each loacation where the
corresponding word index for the word in the review appears.

Next let's write a reusable model-building function that we'll use in all of our experiments in this section.

In [ ]:
def get_model(max_tokens=20000, hidden_dim=16):
    """
    """
    # a simple single dense layer with dropout model
    # our task is a binary classification, so final layer has 1 output
    # using sigmoid activation
    inputs = keras.Input(shape=(max_tokens,))
    x = layers.Dense(hidden_dim, activation="relu")(inputs)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    # create and compile the model for binary classification
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

Finally let's train and test this first model.

In [ ]:
model = get_model()
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("../models/binary_1gram.keras", save_best_only=True)
]

# notice the call to cache() for the dataset, this will cache them in primary RAM memory.
# This way we will do the preprocessing to vectorize only 1 time during epoch 1, and will reuse the
# preprocessed texts for the following epochs.  This can only be done when data is small enough to
# fit into memory
history = model.fit(binary_1gram_train_ds.cache(),
                    validation_data=binary_1gram_val_ds.cache(),
                    epochs=10,
                    callbacks=callbacks)

# reload best seen model by validation loss to test
model = keras.models.load_model("../models/binary_1gram.keras")
print(f"Test acc: {model.evaluate(binary_1gram_test_ds)[1]:.3f}")

This should usually achieve a test accuracy of around 89%.  Not bad.  If you go and look back, the first time we did this on the
IMDB database with 2 layers of 16 units we got about 87% accuracy.  Though there we used only the first 10,000
words instead of 20,000 in a multi-hot encoding, and it may have been overfitting, which might be why we get a bit
better performance here usually with only a single layer.

Note that in this case the dataset is a balanced two-class 
classification dataset, so the naive baseline we could reach without training would only be 50%. Meanwhile
just FYI, the best score that can be achieved on this dataset without leveraging external data is around 95% test accuracy.

#### Bigrams with binary encoding

Of course discarding word order is very reductive, because even atomic concepts can be
expressed via multiple word terms: "United States" conveys a single concept that is differented
from "united" and "states".

For this reason, you will usually end up re-injecting local order information int a bag-of-words representation
by looking at N-grams rather than single words.

The `TextVectorization` layer can be configured to return arbitrary N-grams: bigrams, trigrams, etc.
Just pass an `ngrams=N` argument:

**Note**: It looks like the `keras.layers.TextVectorization` does return 1 and 2-gram by default when using
its `ngrams` parameter, not just the 2-grams.

In [ ]:
# create the TextVectorization instance again, but use
# 2-grams for the vocabulary
text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="multi_hot",
)

# we create the 2-gram vocabulary here
text_vectorization.adapt(text_only_train_ds)

binary_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

binary_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

Just to make sure we are clear here, what does the vocabulary look like now in the
`text_vectorization` instance once we trained on 2-gram input:

In [ ]:
vocabulary = text_vectorization.get_vocabulary()
print(len(vocabulary))
print(vocabulary[:100])

You should see if you look closely that there are 2-word pairs along with the single words in the dictionary.
Also you might want to consider, the number of possible 2-grams has extended the size of the potential
input vocabularly considerably.  I wonder if increasing the `max_tokens` would help here or not.

Let's test how our model performs when trained on such binary-encoded bags of bigrams.  Note because
`max_tokens` is still set to 20,000, the output from our `Dataset`s will still be a multi-hot
vector of 20,000 features.

In [ ]:
model = get_model()
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("../models/binary_2gram.keras", save_best_only=True)
]


model.fit(binary_2gram_train_ds.cache(),
          validation_data=binary_2gram_val_ds.cache(),
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model("../models/binary_2gram.keras")
print(f"Test acc: {model.evaluate(binary_2gram_test_ds)[1]:.3f}")

You should now usually get at or above 90% on test accuracy, which is a definite improvement.  Turns out local
order is pretty important.

#### Bigrams with TF-IDF encoding

You can add a bit more information to this representation by counting how many times each word or 
N-gram occurs in an input sample.  That is to say, by taking histograms of words over the text.

If you are doing text classification, knowing how many times a word occurs in a sample is critical:
any sufficiently long movie review may contain the word "terrible" regardless of sentiment, but a review
that contains many instances of "terrible" is likely a negative one.

Our previous multi-encoding to vectorize the input that we did by hand could easily be modified
to encode the counts of words/bigrams, simply add 1 to the value in the dictionary each time we
see it when encoding.

For the `TextVectorization` layer, simply use the `output_mode="count"` parameter.

In [ ]:
text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="count"
)

Another topic we haven't addressed yet, normalization of texts when doing text processing
tasks.  Small common words like "the", "a", "is" in English will always dominate your
word count histograms, drowning out other words, despite being pretty much
uselss features in a classificaiton context like this.

We could normalize word counts by subtracting the mean and dividing by the variance. Except most
vectorized sentences consist almost entirely of zeros, a property called sparsity.
Using the basic normalization (mean centering and dividing by std) would wreck the sparsity,
which for computational reasons you would like to preserve.

Instead we use **TF-IDF normalization**, which stand for "term frequency inverse document frequency".

TF-IDF is so common that it's built into the `TextVectorization` layer.  All you need to do to
start using it is to switch the `output_mode` to "tf_idf".

In [ ]:
# have the TextVectorization perform the TF-IDF normalization on token histograms in samples
text_vectorization = layers.TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode="tf_idf",
)

# the adapt() call will learn the TF-IDF weights in addition
# to the vocabulary
text_vectorization.adapt(text_only_train_ds)

tfidf_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

tfidf_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

tfidf_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [ ]:
model = get_model()
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint("../models/tfidf_2gram.keras", save_best_only=True)
]

history = model.fit(tfidf_2gram_train_ds.cache(),
                    validation_data=tfidf_2gram_val_ds.cache(),
                    epochs=10,
                    callbacks=callbacks)

model = keras.models.load_model("../models/tfidf_2gram.keras")
print(f"Test acc: {model.evaluate(tfidf_2gram_test_ds)[1]:.3f}")

For the IMDB dataset, you probably won't see any improvement here using TF-IDF.  However for
many text-classification datasets, it would be typical to see a performance increase when
using TF-IDF compared to the plain histogram counts.

#### Exporting a model that processes raw strings

In the preceding examples, we did our text standardization, splitting, and indexing as
part of the tf.data pipeline. But if we want to export a standalone model independent
of this pipeline, we should make sure that it incorporates its own text preprocessing
(otherwise, you’d have to reimplement in the production environment, which
can be challenging or can lead to subtle discrepancies between the training data and
the production data). Thankfully, this is easy in Keras.

Just create a new model that reuses your TextVectorization layer and adds to it
the model you just trained:

In [ ]:
# notice change in Input here, a vector of only 1 element, which is a string data type
inputs = keras.Input(shape=(1,), dtype="string")
processed_inputs = text_vectorization(inputs)
outputs = model(processed_inputs)

inference_model = keras.Model(inputs, outputs)

# the new model has the trained text_vectorization instance in it, before feeding into
# the previous trained model, which expects exactly the output that the text_vectorizaiton does
inference_model.summary()

The resulting model can process batches of raw strings.

In [ ]:
raw_text_data = tf.convert_to_tensor([
    ["That was an excellent movie, I loved it."],
    ["Awful, do not waste your money. Avoid seeing it."],
])

predictions = inference_model(raw_text_data)
print(f"{float(predictions[0] * 100):.2f} percent positive")
print(f"{float(predictions[1] * 100):.2f} percent positive")

### 11.3.3 Processing words as a sequence

These past few examples clearly show that word order matters: manual engineering of
order-based features, such as bigrams, yields a nice accuracy boost.

What if, instead of manually crafting order-based features, we exposed the model to raw word sequences
and let it figure out such features on its own? This is what **sequence models** are about.

To implement a sequence model, you’d start by representing your input samples as
sequences of integer indices (one integer standing for one word). Then, you’d map
each integer to a vector to obtain vector sequences. Finally, you’d feed these
sequences of vectors into a stack of layers that could cross-correlate features from adjacent
vectors, such as a 1D convnet, a RNN, or a Transformer.

Let's try a sequence model.  First we need a dataset that returns integer sequences 
(rather than a multi-hot encoded vector representation).

In [ ]:
# we will create sequences, but will mave the maximum sequence length of 600 tokens.
# this means longer reviews will get choped to first 600 words, and shorts ones will be
# filled with the mask index 0 token
max_length = 600

# but we will still use most frequent 20,000 words for the vocabulary
max_tokens = 20000

# we'll truncate inputs after firt 600 words, this is a reasonable
# choice since the average review is 233 words and only 5%
# of reviews are longer than 600
# also since we are using sequences, bigram doesn't make sense, so we
# go back to 1-gram vocabulary
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)

# build the vocabulary again
text_vectorization.adapt(text_only_train_ds)

# The dataset instances again on the sequence output
# if we looked, what output would you expect from these,
# should be (32, 600) shaped outputs for batch size 32
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [ ]:
# just to be clear what we have now
for inputs, targets in int_train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

Next let's make a model.  We still have to convert integer sequences to
vector sequences.  The simplest way is to one-hot encode the integers (each dimension
would represent one possible term in the vocabulary).  On top of these one-hot
vectors we'll add a simple bidirectional LSTM.

In [ ]:
# also to be clear, if you didn't follow about the one-hot encoding
# if you one hot encode a sample of 600 input integers using 20,000 dimension encoding you get:
tf.one_hot(inputs[0], depth=max_tokens)

In [ ]:
# textbook shows directly using tf.one_hot() instance, but API has changed.
# example of subclassing the keras layer class
class TFOneHotEncodeIntSequence(keras.Layer):
    def call(self, x):
        return tf.one_hot(x, depth=max_tokens)


# one input is a sequence of integers 
inputs = keras.Input(shape=(None,), dtype="int64")
# encode the integers into binary 20,000 dimensional vectors
embedded = TFOneHotEncodeIntSequence()(inputs)
# add a bidirectional LSTM layer
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
# normal binary classification output layer
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Now let's try it out and train this sequence model.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/one_hot_bidir_lstm.keras", save_best_only=True)
]

history = model.fit(int_train_ds, 
                    validation_data=int_val_ds, 
                    epochs=10,
                    callbacks=callbacks)

model = keras.models.load_model("../models/one_hot_bidir_lstm.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

You will first find that this model trains much more slowly than the previous ones.  This is because
the inputs are quite large, each input sample is encoded as a matrix of size `(600, 2000)`
(600 words per sample, 20,000 possible words).  That's 12,000,000 floats for a single movie review.

But if you can get this to train, you will find it doesn't improve performance at all, and it is maybe a bit
worse than the 90% we saw we could get.  Clearly using one-hot encoding to turn words into vectors, which was
the simplest thing we could do, wasn't a great idea.  There's a better way: **word embeddings**.

#### Understanding word embeddings

The fundamental assumption is that the different tokens you’re encoding are all independent from each other: indeed, one-hot vectors are all orthogonal
to one another.

In the case of words that assumption is clearly wrong.  Words form a structured space, they share information with each other.
The words “movie” and “film” are interchangeable in most sentences, so the vector that represents
“movie” should not be orthogonal to the vector that represents “film”—they should be
the same vector, or close enough. The geometric relationship between two word vectors
should reflect the semantic relationship between these words. 

**Word embeddings** are vector representations of words that achieve exactly this: they
map human language into a structured geometric space.

Whereas the vectors obtained through one-hot encoding are binary, sparse (mostly
made of zeros), and very high-dimensional (the same dimensionality as the number of
words in the vocabulary), word embeddings are low-dimensional floating-point vectors
(that is, dense vectors, as opposed to sparse vectors);

Besides being dense representations, word embeddings are also structured representations,
and their structure is learned from data. Similar words get embedded in close
locations, and further, specific directions in the embedding space are meaningful.

Two ways to obtain word embeddings

- Learn word embeddings jointly with the main task you care about (such as document
  classification or sentiment prediction). In this setup, you start with random
  word vectors and then learn word vectors in the same way you learn the
  weights of a neural network.
- Load into your model word embeddings that were precomputed using a different
  machine learning task than the one you’re trying to solve. These are called
  pretrained word embeddings.

#### Learning word embeddings with the `Embedding` layer

What makes a good word-embedding space depends heavily on
your task: the perfect word-embedding space for an English-language movie-review
sentiment-analysis model may look different from the perfect embedding space for an
English-language legal-document classification model.

The importance of certain semantic relationships varies from task to task.

Thus it is reasoanable to **learn** a new embedding space for your task.

Can train a parallel model (using same optimizaiton techniques you are familiar with)
using the Keras `Embedding` layer.


In [ ]:
# the embedding layer takes at least two arguments, the number of possible tokens and the
# dimensionality of the embedding space to create (here 256)
embedding_layer = layers.Embedding(input_dim=max_tokens, output_dim=256)

The Embedding layer is best understood as a dictionary that maps integer indices
(which stand for specific words) to dense vectors. It takes integers as input, looks up
these integers in an internal dictionary, and returns the associated vectors. It’s effectively
a dictionary lookup

Once fully trained, the embedding space will show a lot of structure—a kind of structure specialized for the specific problem
for which you’re training your model.

Let's build a model that includes an `Embedding` layer and benchmark it on our task:

In [ ]:
# similar to before, but instead of an embeded one-hot encoding layer,
# use Embedding layer to learn a word embedding space
# and in fact input is same sequence of int tokens as before
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/embeddings_bidir_lstm.keras", save_best_only=True)
]

model.fit(int_train_ds, 
          validation_data=int_val_ds, 
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model("../models/embeddings_bidir_lstm.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

It trains much faster than the one-hot model (since LSTM only has to process a
256 dimensional vector instead of 20,000 dimensional).  And its text
accuracy is comparable, you will again probably only usually get 87% accuracy again.

Part of the reason may be the truncation to 600 words (according to text).

#### Understanding padding and masking

One thing that’s slightly hurting model performance here is that our input sequences
are full of zeros because most reviews are shorter than 600 words, so they are filled in with the MASK.

Sentences longer than 600 are truncated, and shorter are padded with 0's.

Recall how RNN work.  The RNN that looks at tokens in forward/natural order will spend its
last iterations a lot of times seing only 0's.  The information stored in the internal state
will gradually fade out as it gets exposed to these meaningless inputs.

We need some way to tell the RNN that it should skip these iterations. There’s an
API for that: **masking**.

The Embedding layer is capable of generating a “mask” that corresponds to its
input data. This mask is a tensor of ones and zeros (or True/False booleans), of shape
(batch_size, sequence_length), where the entry mask[i, t] indicates where timestep
t of sample i should be skipped or not (the timestep will be skipped if mask[i, t]
is 0 or False, and processed otherwise).

By default this option is off, you can turn it on by using `mazk_zero=True`
to your `Embedding` layer.

In [ ]:
# example on made up input of what the embedding mask looks like
embedding_layer = layers.Embedding(input_dim=10, output_dim=256, mask_zero=True)

# made up input
some_input = [
    [4, 3, 2, 1, 0, 0, 0],
    [5, 4, 3, 2, 1, 0, 0],
    [2, 1, 0, 0, 0, 0, 0]]

mask = embedding_layer.compute_mask(some_input)
mask

In practice you don't have to manage masks by hand.  Instead can automatically pass on the
mask to every layer that is able to process it.  This mask will be used by RNN layers
to skip masked steps.  

Let's retrain the previous mask with masking enabled this time.

In [ ]:
# same as before, but try out using mask_zero=True in our word Embedding layer
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256, mask_zero=True)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/embeddings_bidir_lstm_with_masking.keras", save_best_only=True)
]

model.fit(int_train_ds, validation_data=int_val_ds, epochs=10, callbacks=callbacks)

model = keras.models.load_model("../models/embeddings_bidir_lstm_with_masking.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

This time you should usually get a bit better than the most previous example without masking, though again still not better than
best model we have seen so far.

#### Using pretrained word embeddings

Sometimes you have so little training data available that you can’t use your data alone
to learn an appropriate task-specific embedding of your vocabulary.  In such cases,
it might be useful to use a pretrained embedding space (similar to how we used
pretrained models for computer vision previously).

To do this we have to download a pretrained embedding (these are not available directly
in the Keras library like the pretrained models we used before).

First let's download the GloVe word embeddings precomputed on the 2014 English
Wikipedia dataset.  It's an 822M zip file containing 100-dimensional embedding vectors for
400,000 words.

For the following examples I expect the files to be in the `../data` subdirectory.  Here is how to
get the embeddings in our class DevContainer environment:

**Note**: The certificate is expired on the url given here in the text.  Can use the `--no-check-certificate` flag to get,
though probably need to find if/where this data might be now.

```
vscode ➜ /workspaces/nndl/data (main) $ wget --no-check-certificate http://nlp.stanford.edu/data/glove.6B.zip

Saving to: ‘glove.6B.zip’
glove.6B.zip                       100%[=============================================================>] 822.24M  4.79MB/s    in 2m 49s  
2025-06-14 22:31:20 (4.87 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]


vscode ➜ /workspaces/nndl/data (main) $ unzip -q glove.6B.zip 
```

The result are several plain text files, named things like `glove.6B.100d.txt` where the `100d` indicates the number of embedding dimensions the
file contains.  These are rather big files (the `300d` file is about 1G unzipped).  We are going to use the `100d` 100 dimensions version,
so to save space I removed the zip file and all the other files:

```
vscode ➜ /workspaces/nndl/data (main) $ rm glove.6B.zip glove.6B.200d.txt glove.6B.300d.txt glove.6B.50d.txt 

```

The file is in a plain text format, so we need a little custom code to read it in.  We start by creating a dictionary that
maps each word to a set of coefficients (its embedding vector).

In [ ]:
# location of the file to open and read in
path_to_glove_file = "../data/glove.6B.100d.txt"

# result of this first part is a simple dictionary that maps an english word
# to a key of the vector coefficients representing the vector encoding of this word.
embeddings_index = {}

with open(path_to_glove_file) as f:
    # each line of file first containes the word, followed by 100 coefficients
    for line in f:
        word, coefs = line.split(maxsplit=1)
        # this creates an numpy array, splitting by space separator
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs
print(f"Found {len(embeddings_index)} word vectors.")

In [ ]:
# As an example, how is the word "movie" represented in this embedding.
print(embeddings_index['movie'])
print(type(embeddings_index['movie']))
print(embeddings_index['movie'].shape)

Next lets build an embedding matrix that you can load into an `Embedding` layer.
It must be a matrix of shape `(max_words, embedding_dim)` where each entry `i`
contains the `embedding_dim` dimensional vector for the word of index `i` in the
reference word index.

In [ ]:
embedding_dim = 100

# retrieve the vocabulary indexed by our previous TextVectorization layer
# remember this is a list, orderd by the word index
vocabulary = text_vectorization.get_vocabulary()
# use it to create a mapping from words to their index in the vocabulary
# this is a dictionary with the word as key and its index as the value now
word_index = dict(zip(vocabulary, range(len(vocabulary))))

# the resulting embedding matrix has to have shape (max_tokens, embedding_dim)
# which in this particular case is (20000, 100) since we used 20,000 words and
# are reusing the 100 dimension word embedding
embedding_matrix = np.zeros((max_tokens, embedding_dim))

for word, i in word_index.items():
    # fill entry i in the matrix with word vector for index i
    # words not found in the embedding (not likely) will be all zeros
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

Finally we use a `Constant` initializer to load the pretrained embeddings in
an Embedding layer.  So as not to disrupt the pretrained representations during training,
we freeze the layer via `trainable=False`

In [ ]:
embedding_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True,
)  

We're now ready to traina new model, identical to our previous model, but leveraging the 100-dimensional pretrained GloVe embeddings instead of 256-dimensional
learned embeddings.

In [ ]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = embedding_layer(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
loss="binary_crossentropy",
metrics=["accuracy"])

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/glove_embeddings_sequence_model.keras", save_best_only=True)
]

model.fit(int_train_ds, 
          validation_data=int_val_ds,
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model("../models/glove_embeddings_sequence_model.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

You'll find that on this particular task, pretrained embeddings aren't very helpful because the dataset
contains enough samples that it si possible to learn a specialized enough embedding space from scratch.
However, leveraging pretrained embeddings can be very helpful when you're working with a smaller dataset.

## Summary

<font color='blue'>
    
- Word order in Text processing is handled in 2 basic ways:
  1. **bag-of-words models** : discard order and treat as an unordered set (multi-hot encoding typically, 20,000 sparse vectors).
  2. **sequence models**: process words in order they appear like a timeseries
- For sequence models, can encode sequence again using one-hot encoding, but this ends up with very large input to RNN models.
- **word embeddings** are vector representations of words that map humanlanguage into a structured geometric space.